In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("imsparsh/lakh-midi-clean")

print("Path to dataset files:", path)

c:\Users\megan\AppData\Local\Programs\Python\Python311\Lib\site-packages\requests\__init__.py:109: RequestsDependencyWarning: urllib3 (2.0.4) or chardet (7.4.0.post2)/charset_normalizer (3.2.0) doesn't match a supported version!
  warnings.warn(


Path to dataset files: C:\Users\megan\.cache\kagglehub\datasets\imsparsh\lakh-midi-clean\versions\1


In [2]:
import data_processing as dp

# Using process_data to get note and duration
# Note: These midi files contain multiple voices so those would need to be accounted for

# Uncomment:
# dp.process_data(midi_dir = path, output_dir="lakh_wav_notes_dir")

In [ ]:
N_BINS = 84
FIXED_LENGTH = 128

def create_cqt(y, sr, hop_length=512):
    cqt = librosa.cqt(y, sr=sr, hop_length=hop_length, n_bins=N_BINS, bins_per_octave=12)
    cqt_db = librosa.amplitude_to_db(np.abs(cqt), ref=np.max)
    return cqt_db  # should be (84, time_steps)

def pad_or_truncate(cqt, fixed_length=FIXED_LENGTH):
    if cqt.shape[1] >= fixed_length:
        return cqt[:, :fixed_length]
    pad_width = fixed_length - cqt.shape[1]
    return np.pad(cqt, ((0, 0), (0, pad_width)))  # should be (84, 128)

In [3]:
import data_processing as dp
import librosa
import pretty_midi
import os
import glob
import numpy as np
import pandas as pd
from multiprocessing import Pool, cpu_count

# Process all the notes at a given time for the audio
def process_single_midi(args):
    # print("In thread")
    midi_path, output_dir, soundfont_path = args
    features, labels = [], []

    try:
        # print(f"Processing midi file: {midi_path}", flush=True)
        wav_path, val = dp.midi_to_wav(midi_path, output_dir, soundfont_path)
        y, sr = librosa.load(wav_path, sr=22050)
        midi_data = pretty_midi.PrettyMIDI(midi_path)

        # Define your time window size (e.g. 0.5 seconds)
        window_size = 0.5
        hop_size = 0.25  # how much to slide the window each step
        total_duration = len(y) / sr
        
        t = 0.0
        while t + window_size <= total_duration:
            # Slice the mixed audio at this time window
            start_sample = int(t * sr)
            end_sample = int((t + window_size) * sr)
            segment = y[start_sample:end_sample]

            # 2. Get all notes across different voices
            active_notes = []
            for instrument in midi_data.instruments:
                for note in instrument.notes:
                    # note overlaps with this window
                    if note.start < (t + window_size) and note.end > t:
                        active_notes.append(pretty_midi.note_number_to_name(note.pitch))

            if len(active_notes) == 0:
                t += hop_size
                continue

            # 3. Compute spectrogram of the mixed window
            cqt = dp.create_cqt(segment, sr)
            cqt_fixed = dp.pad_or_truncate(cqt)

            features.append(cqt_fixed)
            labels.append(active_notes) 
            
            t += hop_size
        print(f"Finished processing midi file: {midi_path}")

    except Exception as e:
        print(f"Error processing {midi_path}: {e}")

    return features, labels

In [4]:
def process_data(midi_dir, with_pool = False):

    # def get_midi_tempo(midi_path):
    #     try:
    #         midi_data = pretty_midi.PrettyMIDI(midi_path)
    #         tempo_times, tempo_values = midi_data.get_tempo_changes()
    #         return tempo_values[0] if len(tempo_values) > 0 else 120.0
    #     except:
    #         return None

    # def filter_midi_by_tempo(midi_files, target_bpm=120, tolerance=5):
    #     """Keep only files within target_bpm +/- tolerance"""
    #     filtered = []
    #     for f in midi_files:
    #         bpm = get_midi_tempo(f)
    #         if bpm is not None and abs(bpm - target_bpm) <= tolerance:
    #             filtered.append(f)
    #     return filtered

    # # usage
    # midi_files = glob.glob(os.path.join(midi_dir, "**/*.mid"), recursive=True)
    # midi_files = filter_midi_by_tempo(midi_files, target_bpm=120, tolerance=5)
    # print(f"Found {len(midi_files)} files at ~120 BPM")

    midi_files = glob.glob(os.path.join(midi_dir, "**/*.mid"), recursive=True)
    print(f"Found {len(midi_files)} MIDI files")
    print(f"Using {cpu_count()} CPU cores")

    args = [(f, output_dir, soundfont_path) for f in midi_files[:100]]
    print(f"Processing {len(args)} midi files")

    all_features, all_labels = [], []

    def pad_features(feature, max_len):
        if len(feature) < max_len:
            # pad with zeros at the end
            return np.pad(feature, (0, max_len - len(feature)))
        else:
            # truncate if too long
            return feature[:max_len]


    if with_pool:
        from concurrent.futures import ThreadPoolExecutor, as_completed
        print("Processing with pooling")

        num_cpu = 2 # cpu_count()

        with ThreadPoolExecutor(max_workers=num_cpu) as executor:
            futures = [executor.submit(process_single_midi, arg) for arg in args]
            for fut in as_completed(futures):
                features, labels = fut.result()
                all_features.extend(features)
                all_labels.extend(labels)
        # with Pool(processes=cpu_count()) as pool:
        #     for i, (features, labels) in enumerate(pool.imap(process_single_midi, args)):
        #         all_features.extend(features)
        #         all_labels.extend(labels)
        #         if (i + 1) % 100 == 0:
                    # print(f"Processed {i+1}/{len(midi_files)} files...")
    else:
        for i, arg in enumerate(args):
            features, labels = process_single_midi(arg)
            all_features.extend(features)
            all_labels.extend(labels)
            if (i + 1) % 10 == 0:
                print(f"Processed {i+1}/{len(args)} files...")

    # Apply before stacking
    all_features = [pad_features(f, dp.FIXED_LENGTH) for f in all_features]

    return all_features, all_labels
    
def save_dataprocessing(all_features, all_labels, save_features, save_labels):
    features = np.stack(all_features)
    df = pd.DataFrame({'labels': all_labels})

    np.save(save_features, all_features)
    df.to_csv(save_labels, index=False)

    print(f"Saved {len(df)} notes")
    print(f"Features shape: {features.shape}")
    print(df['labels'].value_counts())

In [ ]:
# Process each midi files into wav

# import data_processing as dp

# all_features, all_labels = process_data(midi_dir = path)

Found 17219 MIDI files
Using 16 CPU cores
Processing 5 midi files
Successfully created: lakh_wav\A_Campfire_Song.wav


c:\Users\megan\AppData\Local\Programs\Python\Python311\Lib\site-packages\pretty_midi\pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(
c:\Users\megan\AppData\Local\Programs\Python\Python311\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=173
  warnings.warn(


Finished processing midi file: C:\Users\megan\.cache\kagglehub\datasets\imsparsh\lakh-midi-clean\versions\1\10,000_Maniacs\A_Campfire_Song.mid
Successfully created: lakh_wav\Theme_From_The_Godfather.wav
Finished processing midi file: C:\Users\megan\.cache\kagglehub\datasets\imsparsh\lakh-midi-clean\versions\1\101_Strings\Theme_From_The_Godfather.mid
Successfully created: lakh_wav\Dreadlock_Holiday.1.wav
Finished processing midi file: C:\Users\megan\.cache\kagglehub\datasets\imsparsh\lakh-midi-clean\versions\1\10cc\Dreadlock_Holiday.1.mid
Successfully created: lakh_wav\Dreadlock_Holiday.2.wav
Finished processing midi file: C:\Users\megan\.cache\kagglehub\datasets\imsparsh\lakh-midi-clean\versions\1\10cc\Dreadlock_Holiday.2.mid
Successfully created: lakh_wav\Dreadlock_Holiday.3.wav
Finished processing midi file: C:\Users\megan\.cache\kagglehub\datasets\imsparsh\lakh-midi-clean\versions\1\10cc\Dreadlock_Holiday.3.mid
Saved 4888 notes
Features shape: (4888, 128, 172)
labels
[F#3]          

In [6]:
output_dir="lakh_wav_pool"
soundfont_path = "GeneralUser-GS/GeneralUser-GS.sf2"

In [6]:
all_features, all_labels = process_data(midi_dir = path, with_pool=True)

Found 17219 MIDI files
Using 16 CPU cores
Processing 100 midi files
Successfully created: lakh_wav_pool\Simon_Says.wav
Successfully created: lakh_wav_pool\Short_Dick_Man.wav
Successfully created: lakh_wav_pool\Theme_From_The_Godfather.wav
Successfully created: lakh_wav_pool\Lick_It.wav
Successfully created: lakh_wav_pool\The_Things_We_Do_for_Love.wav
Successfully created: lakh_wav_pool\Simon_Says.1.wav
Successfully created: lakh_wav_pool\A_Campfire_Song.wav
Successfully created: lakh_wav_pool\Im_Not_In_Love.2.wav
Successfully created: lakh_wav_pool\Im_Not_In_Love.1.wav
Successfully created: lakh_wav_pool\Im_Not_In_Love.3.wav
Successfully created: lakh_wav_pool\Im_Not_In_Love.wav


c:\Users\megan\AppData\Local\Programs\Python\Python311\Lib\site-packages\pretty_midi\pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(
c:\Users\megan\AppData\Local\Programs\Python\Python311\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=173
  warnings.warn(


Successfully created: lakh_wav_pool\Dreadlock_Holiday.4.wav
Successfully created: lakh_wav_pool\Dreadlock_Holiday.3.wav
Successfully created: lakh_wav_pool\Dreadlock_Holiday.2.wav
Successfully created: lakh_wav_pool\Dreadlock_Holiday.wav
Successfully created: lakh_wav_pool\Dreadlock_Holiday.1.wav
Successfully created: lakh_wav_pool\I_Wont_Let_You_Down.wav
Finished processing midi file: C:\Users\megan\.cache\kagglehub\datasets\imsparsh\lakh-midi-clean\versions\1\101_Strings\Theme_From_The_Godfather.mid
Successfully created: lakh_wav_pool\Come_Take_My_Hand.wav
Finished processing midi file: C:\Users\megan\.cache\kagglehub\datasets\imsparsh\lakh-midi-clean\versions\1\1910_Fruitgum_Company\Simon_Says.mid
Successfully created: lakh_wav_pool\Dreams.1.wav
Finished processing midi file: C:\Users\megan\.cache\kagglehub\datasets\imsparsh\lakh-midi-clean\versions\1\1910_Fruitgum_Company\Simon_Says.1.mid
Successfully created: lakh_wav_pool\Dreams.wav
Finished processing midi file: C:\Users\megan\.

OSError: data byte must be in range 0..127

In [7]:
output_dir="lakh_wav_pool"
all_features, all_labels = process_data(midi_dir = path, with_pool=True)

Found 17219 MIDI files
Using 16 CPU cores
Processing 100 midi files
Processing with pooling
Successfully created: lakh_wav_pool\Theme_From_The_Godfather.wav
Successfully created: lakh_wav_pool\A_Campfire_Song.wav


c:\Users\megan\AppData\Local\Programs\Python\Python311\Lib\site-packages\pretty_midi\pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(
c:\Users\megan\AppData\Local\Programs\Python\Python311\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=173
  warnings.warn(


Finished processing midi file: C:\Users\megan\.cache\kagglehub\datasets\imsparsh\lakh-midi-clean\versions\1\101_Strings\Theme_From_The_Godfather.mid
Finished processing midi file: C:\Users\megan\.cache\kagglehub\datasets\imsparsh\lakh-midi-clean\versions\1\10,000_Maniacs\A_Campfire_Song.mid
Successfully created: lakh_wav_pool\Dreadlock_Holiday.1.wav
Successfully created: lakh_wav_pool\Dreadlock_Holiday.2.wav
Finished processing midi file: C:\Users\megan\.cache\kagglehub\datasets\imsparsh\lakh-midi-clean\versions\1\10cc\Dreadlock_Holiday.1.mid
Successfully created: lakh_wav_pool\Dreadlock_Holiday.3.wav
Finished processing midi file: C:\Users\megan\.cache\kagglehub\datasets\imsparsh\lakh-midi-clean\versions\1\10cc\Dreadlock_Holiday.3.mid
Successfully created: lakh_wav_pool\Dreadlock_Holiday.4.wav
Error processing C:\Users\megan\.cache\kagglehub\datasets\imsparsh\lakh-midi-clean\versions\1\10cc\Dreadlock_Holiday.4.mid: data byte must be in range 0..127
Successfully created: lakh_wav_pool\

In [8]:
save_dataprocessing(all_features, all_labels, 'features_multilabel.npy', 'labels_multilabel.csv')

Saved 3750 notes
Features shape: (3750, 128, 172)
labels
[D4, B3, G3, G2, D4, D4, B3, G3, G2]                                                                                                                                                             13
[G#3, G#4, G#5, B3, B4, B5, G#5, B5, C3, D3, C5, G#4, F1, F#3, F#3, E2]                                                                                                                          12
[D4, A3, D2, D4, D4, A3, D2]                                                                                                                                                                     12
[B3, D4, G2, D4, B3, D4, G2]                                                                                                                                                                     10
[F#4, C#4, A3, E4, C#4, A3, A2, F#4, E4, F#4, C#4, A3, E4, C#4, A3, A2]                                                                                        

# Training Model to process multilabel

In [9]:
import dataset
# Prepare the data
train_dataset, val_dataset, test_dataset, num_classes = dataset.data_pipeline_multilabel(feature_id="labels", features="features_1.npy", labels="labels_1.csv")

['A#1' 'A#2' 'A#3' 'A#4' 'A#5' 'A1' 'A2' 'A3' 'A4' 'A5' 'B1' 'B2' 'B3'
 'B4' 'B5' 'C#2' 'C#3' 'C#4' 'C#5' 'C2' 'C3' 'C4' 'C5' 'C6' 'D#2' 'D#3'
 'D#4' 'D#5' 'D#6' 'D2' 'D3' 'D4' 'D5' 'D6' 'E1' 'E2' 'E3' 'E4' 'E5' 'E6'
 'F#1' 'F#2' 'F#3' 'F#4' 'F#5' 'F1' 'F2' 'F3' 'F4' 'F5' 'F6' 'G#1' 'G#2'
 'G#3' 'G#4' 'G#5' 'G1' 'G2' 'G3' 'G4' 'G5' 'G6']
(4888, 62)
[0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0
 0 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Total notes classes: 2
Creating a new dataloader to: ./dataloader/train.pt, ./dataloader/val.pt, ./dataloader/test.pt


In [24]:
from torch.utils.data import DataLoader, Dataset
import torch
class MultiNoteDataset(Dataset):
    def __init__(self, features, note_labels, num_classes):
        self.note_labels = note_labels
        self.features = features
        self.num_classes = num_classes

    def __len__(self):
        return len(self.note_labels)

    def __getitem__(self, idx):
        feature = torch.tensor(self.features[idx], dtype=torch.float32).unsqueeze(0)
        label = torch.tensor(self.note_labels[idx], dtype=torch.float32)  # already multi-hot
        return feature, label

In [25]:
import torch
# from torch.utils.data import Dataset
# from torchvision.io import read_image
import os
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import LabelEncoder
import ast
from sklearn.preprocessing import MultiLabelBinarizer
# Load dataset from features and files
features = np.load("features_multilabel.npy")    # (n, 84, 128) for 84 frequency bins and 128 time steps
df = pd.read_csv('labels_multilabel.csv')

split = [0.8, 0.1, 0.1]
dataloader_train_path = "multilabel_train_dataloader"
dataloader_val_path = "multilabel_val_dataloader"
dataloader_test_path = "multilabel_test_dataloader"

# Encode labels to numeric representation
le = LabelEncoder()

# if labels, 
# get labels of specific feature/y label
feature_id = 'labels'
try: 
    df['parsed_labels'] = df[feature_id].apply(ast.literal_eval)
    mlb = MultiLabelBinarizer()
    # labels = le.fit_transform(df[feature_id].values)
    labels = mlb.fit_transform(df['parsed_labels'].values)
    print(mlb.classes_)        # all 61 possible notes
    print(labels.shape)        # (n_frames, 61)
    print(labels[0])

    labels = torch.tensor(labels, dtype=torch.float32)
except Exception as e:
    print(f"Error encoding labels for feature_id '{feature_id}': {e}.\nAvailable columns in labels: {df.columns.tolist()}")
    raise ValueError(f"Invalid feature_id '{feature_id}'. Please check the labels CSV file for available columns.")
# duration_labels = le.fit_transform(df['duration'].values)       
num_labels = len(np.unique(labels)) # TODO: change number of labels
# num_durations = len(duration_labels.unique())  

print(f"Total notes classes: {num_labels}")
# print(f"Unique duration classes: {num_durations}")
# print(f"Classes: {le.classes_}")

print(f"Creating a new dataloader to: {dataloader_train_path}, {dataloader_val_path}, {dataloader_test_path}")
dataset = MultiNoteDataset(features, labels, num_classes=61)  # Use note labels for training the note model

# Train/val split
train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
    dataset, split
)

    # Save the dataloaders --> Too large to load
    # torch.save(train_dataset, dataloader_train_path)
    # torch.save(val_dataset, dataloader_val_path)
    # torch.save(test_dataset, dataloader_test_path)

# create loaders
dataloader_train = DataLoader(train_dataset, batch_size=32, shuffle=True)
dataloader_val = DataLoader(val_dataset, batch_size=32)
dataloader_test = DataLoader(test_dataset, batch_size=32)

['A#1' 'A#2' 'A#3' 'A#4' 'A#5' 'A1' 'A2' 'A3' 'A4' 'A5' 'B1' 'B2' 'B3'
 'B4' 'B5' 'C#2' 'C#3' 'C#4' 'C#5' 'C2' 'C3' 'C4' 'C5' 'C6' 'D#2' 'D#3'
 'D#4' 'D#5' 'D#6' 'D2' 'D3' 'D4' 'D5' 'D6' 'E1' 'E2' 'E3' 'E4' 'E5' 'E6'
 'F#1' 'F#2' 'F#3' 'F#4' 'F#5' 'F1' 'F2' 'F3' 'F4' 'F5' 'F6' 'G#1' 'G#2'
 'G#3' 'G#4' 'G#5' 'G1' 'G2' 'G3' 'G4' 'G5' 'G6']
(3750, 62)
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0]
Total notes classes: 2
Creating a new dataloader to: multilabel_train_dataloader, multilabel_val_dataloader, multilabel_test_dataloader


In [27]:
from torch import nn
# https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html
class MultiCNN(nn.Module):
    def __init__(self, num_notes, kernel_size=(3, 3)):
        super().__init__()

        self.kernel_size = kernel_size

        self.conv = nn.Sequential(
            # Use a kernel size
            nn.Conv2d(1, 32, kernel_size=self.kernel_size, stride=1, padding=1), # input channel (1 for grayscale), output channels
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=self.kernel_size, padding=1), # input channels -> 32 (from previous conv layer)
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=self.kernel_size, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.flatten = nn.Flatten()

        # Layer to classify the notes
        self.note_classification = nn.Sequential(
            nn.Linear(128 * 16 * 21, 256),  # 43008 instead of 20480
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_notes),
        )



    def forward(self, x):
        x = self.conv(x)
        x = self.flatten(x)
        logits = self.note_classification(x)
        return logits
    

In [ ]:
dataloader_train.

In [ ]:
from torch import nn
import torch
from tqdm import tqdm

# First iteration of training using Adam optimizer
train_losses = [] # keep train of loss

from model import CNN

model = MultiCNN(num_notes=62) # todo: get this number from the dataset
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
correct = 0
total = 0

train_dataset = dataloader_train
print("Starting training loop...")

for epoch in range(2):  # loop over the dataset multiple times
    model.train()

    running_loss = 0.0
    for i, data in tqdm(enumerate(train_dataset, 0), total=len(train_dataset)):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data
        labels = labels.float() 

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.numel()

        # print statistics
        running_loss += loss.item()
        if i % 50 == 49:    
            avg_loss = running_loss / 50
            train_losses.append(avg_loss)
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
            running_loss = 0.0

print('Finished Training')
torch.save(model.state_dict(), "cnn_model.pth")

print(f'Training accuracy: {correct / total:.4f}')


Starting training loop...


  0%|          | 0/94 [00:00<?, ?it/s]C:\Users\megan\AppData\Local\Temp\ipykernel_38860\1798569958.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  label = torch.tensor(self.note_labels[idx], dtype=torch.float32)  # already multi-hot
100%|██████████| 94/94 [00:45<00:00,  2.07it/s]

Finished Training
Training accuracy: 0.8142


# Exploring IOI's for building HMMS

In [18]:
# Be able to 

import librosa
import numpy as np
def get_iois(audio_path):
    audio, sr = librosa.load(audio_path, sr=None)
    print(audio)
    # audio = audio / np.max(np.abs(audio))
    # print(audio)
    # Detect onset frames
    onset_frames = librosa.onset.onset_detect(
        y=audio, sr=sr, units='time'
    )
    # IOIs are just gaps between consecutive onsets
    iois = np.diff(onset_frames)  # e.g. [0.48, 0.25, 0.24, 0.51]
    return iois

In [19]:
# get iois of wav file


iois = get_iois("./lakh_wav/A_Campfire_Song.wav")

[-3.0517578e-05 -3.0517578e-05 -1.5258789e-05 ... -3.0517578e-05
 -3.0517578e-05 -3.0517578e-05]


In [21]:
iois = get_iois("./lakh_wav/Theme_From_The_Godfather.wav")
print(iois)

[-3.0517578e-05 -3.0517578e-05 -1.5258789e-05 ... -3.0517578e-05
 -3.0517578e-05 -3.0517578e-05]
[0.38312925 0.37151927 0.38312925 0.3599093  1.50929705 0.37151927
 0.37151927 0.38312925 0.37151927 0.37151927 0.12770975 0.12770975
 0.11609977 0.38312925 0.37151927 0.37151927 0.37151927 0.38312925
 0.37151927 0.37151927 0.37151927 0.2554195  0.12770975 0.12770975
 0.24380952 0.38312925 0.37151927 0.37151927 0.37151927 0.38312925
 0.37151927 0.37151927 0.37151927 0.38312925 0.37151927 0.37151927
 0.38312925 0.37151927 0.37151927 0.2554195  0.12770975 0.12770975
 0.24380952 0.37151927 0.37151927 0.38312925 0.37151927 0.37151927
 0.38312925 0.37151927 0.37151927 0.38312925 0.37151927 0.37151927
 0.37151927 0.38312925 0.37151927 0.37151927 0.38312925 0.37151927
 0.37151927 0.38312925 0.37151927 0.37151927 0.37151927 0.38312925
 0.37151927 0.37151927 0.38312925 0.2554195  0.11609977 0.12770975
 0.24380952 0.37151927 0.38312925 0.37151927 0.37151927 0.38312925
 0.37151927 0.37151927 0.3831292

In [6]:
# Process each midi files into wav

# import data_processing as dp

process_data(midi_dir = path, output_dir="lakh_wav")

Found 17219 MIDI files
Using 16 CPU cores
Processing 5 midi files
In thread
Processing midi file: C:\Users\megan\.cache\kagglehub\datasets\imsparsh\lakh-midi-clean\versions\1\10,000_Maniacs\A_Campfire_Song.mid
Successfully created: lakh_wav\A_Campfire_Song.wav


c:\Users\megan\AppData\Local\Programs\Python\Python311\Lib\site-packages\pretty_midi\pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(
c:\Users\megan\AppData\Local\Programs\Python\Python311\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=256 is too large for input signal of length=173
  warnings.warn(


Finished processing midi file: C:\Users\megan\.cache\kagglehub\datasets\imsparsh\lakh-midi-clean\versions\1\10,000_Maniacs\A_Campfire_Song.mid
In thread
Processing midi file: C:\Users\megan\.cache\kagglehub\datasets\imsparsh\lakh-midi-clean\versions\1\101_Strings\Theme_From_The_Godfather.mid
Successfully created: lakh_wav\Theme_From_The_Godfather.wav
Finished processing midi file: C:\Users\megan\.cache\kagglehub\datasets\imsparsh\lakh-midi-clean\versions\1\101_Strings\Theme_From_The_Godfather.mid
In thread
Processing midi file: C:\Users\megan\.cache\kagglehub\datasets\imsparsh\lakh-midi-clean\versions\1\10cc\Dreadlock_Holiday.1.mid
Successfully created: lakh_wav\Dreadlock_Holiday.1.wav
Finished processing midi file: C:\Users\megan\.cache\kagglehub\datasets\imsparsh\lakh-midi-clean\versions\1\10cc\Dreadlock_Holiday.1.mid
In thread
Processing midi file: C:\Users\megan\.cache\kagglehub\datasets\imsparsh\lakh-midi-clean\versions\1\10cc\Dreadlock_Holiday.2.mid
Successfully created: lakh_wav